In [1]:
from __future__ import annotations
import operator
from pathlib import Path
from typing import List, Literal,TypedDict,Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import  HumanMessage, SystemMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_groq import ChatGroq



## Defining Schemas

#### Task Model

In [2]:
class Task(BaseModel):
    id: int
    title: str
    goal: str = Field(..., description="One sentence describing what the user is able to do after this section")
    bullets: List[str] = Field(...,
                               min_length=3,
                                 max_length=6,
         description="3-6 concrete, non-overlapping subpoints to cover this section")
    target_words: int = Field(..., description="The target number of words for this section is (120-550)")
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False


#### Evidence Item Schema

In [3]:
class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at : Optional[str] = None # keep tavilyt provides, don't rely on it 
    source: Optional[str] = None # keep tavilyt provides, don't rely on it
    snippet: Optional[str] = None # keep tavilyt provides, don't rely on it

#### Evidence Pack Schema

In [4]:
class EvidencePack(BaseModel):
   evidence: List[EvidenceItem] = Field(default_factory=list)

#### Router Decision Schema

In [5]:
class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal['closed_book', 'hybrid', 'open_book']
    queries : List[str] = Field(default_factory=list)

## Defining State

In [6]:
class State(TypedDict):
    topic: str

    # Routing/Research
    mode: str
    need_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]

    # workers
    sections: Annotated[List[tuple[int,str]],operator.add] # taskid,section_md
    final:  str   

In [7]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    
)

## Defining the Router Node

In [8]:
router_system = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false):
  Evergreen topics where correctness does not depend on recent facts (concepts, fundamentals).
- hybrid (needs_research=true):
  Mostly evergreen but needs up-to-date examples/tools/models to be useful.
- open_book (needs_research=true):
  Mostly volatile: weekly roundups, "this week", "latest", rankings, pricing, policy/regulation.

If needs_research=true:
- Output 3–10 high-signal queries.
- Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
- For open_book weekly roundup, include queries that reflect the last 7 days constraint.
"""

def RoterNode(state: State) -> dict:
    topic = state['topic']
    decider = llm.with_structured_output(RouterDecision)
    decision = decider.invoke(
        [
            SystemMessage(content=router_system),
            HumanMessage(content = {'Topic': topic})
        ]
    )
    return{
        "needs_research": decision.needs_research,
        "mode": decision.mode,
        "queries": decision.queries
    }

### Function for conditional edge

In [9]:
def route_next(state: State) ->str:
    return "research" if state['need_research'] else 'orchestrator' # agr research ki zroort ho to reasearch k node pr jao aur agr nhi to orchestrator k node pr

## Defining Research Node

In [11]:
def tavily_search(query:str, max_results: int = 5) ->List[dict]:
    tool = tavily_search(max_results=max_results)
    tool.invoke({"query": query})

    normalized : List[dict] = []

    for r in results:
        normalized.append(
            {
                "title": r.get("title") or "",
                "url": r.get("url") or "",
                "snippet": r.get('content') or r.get("snippet") or "",
                "published_at": r.get("published_date") or r.get("published_at"),
                "source": r.get("source"),

            }
        )  

    return normalized

RESEARCH_SYSTEM = """You are a research synthesizer for technical writing.

Given raw web search results, produce a deduplicated list of EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer relevant + authoritative sources (company blogs, docs, reputable outlets).
- If a published date is explicitly present in the result payload, keep it as YYYY-MM-DD.
  If missing or unclear, set published_at=null. Do NOT guess.
- Keep snippets short.
- Deduplicate by URL.
""" 

def research_node(state: State) ->dict:
    queries = state.get("queries") or [] # get first 10 queries from state
    max_result = 6
    raw_results : List[dict] = []

    for q in queries:
        raw_results.extend(tavily_search(q,max_result))
    if not raw_results:
        return {"evidence": []}
    
    extractor = llm.with_structured_output(EvidencePack)
    pack = extractor.invoke([
        SystemMessage(content=RESEARCH_SYSTEM),
        HumanMessage(content=f"Raw Results are: {raw_results}")]
    )

    # Deduplicate by URL (“Ye code duplicate URLs ko remove kar raha hai aur sirf unique URLs wali evidence return kar raha hai.”)
    dedup = {}
    for e in pack.evidence:
        if e.url:
            dedup[e.url] = e
  
    return {'evidence': list(dedup.values())}

## Defining Orchestrator Node

In [12]:
ORCH_SYSTEM = """You are a senior technical writer and developer advocate.
Your job is to produce a highly actionable outline for a technical blog post.

Hard requirements:
- Create 5–9 sections (tasks) suitable for the topic and audience.
- Each task must include:
  1) goal (1 sentence)
  2) 3–6 bullets that are concrete, specific, and non-overlapping
  3) target word count (120–550)

Quality bar:
- Assume the reader is a developer; use correct terminology.
- Bullets must be actionable: build/compare/measure/verify/debug.
- Ensure the overall plan includes at least 2 of these somewhere:
  * minimal code sketch / MWE (set requires_code=True for that section)
  * edge cases / failure modes
  * performance/cost considerations
  * security/privacy considerations (if relevant)
  * debugging/observability tips

Grounding rules:
- Mode closed_book: keep it evergreen; do not depend on evidence.
- Mode hybrid:
  - Use evidence for up-to-date examples (models/tools/releases) in bullets.
  - Mark sections using fresh info as requires_research=True and requires_citations=True.
- Mode open_book:
  - Set blog_kind = "news_roundup".
  - Every section is about summarizing events + implications.
  - DO NOT include tutorial/how-to sections unless user explicitly asked for that.
  - If evidence is empty or insufficient, create a plan that transparently says "insufficient sources"
    and includes only what can be supported.

Output must strictly match the Plan schema.
"""

def orchestrator(state: State) -> dict:
    planner = llm.with_structured_output(Plan)

    evidence = state.get('evidence')
    mode = state.get('mode','closed_book')

    plan = planner.invoke([
        SystemMessage(content=ORCH_SYSTEM),
        HumanMessage(content=
                     f"Topic: {state['topic']}\n"
                    f"Mode: {mode}\n\n"
                    f"Evidence (ONLY use for fresh claims; may be empty):\n"
                    f"{[e.model_dump() for e in evidence][:16]}" # Sirf first 16 items le raha hai list ke
                    )
    ]
        
    )
    return{"plan": plan}